# Práctica Final NLP
## Notebook 2: Preprocesado

Después de la exploración ahora realizo el preprocesado, centrandome en eliminar stopwords, lematizar para reducir la cardinalidad de 26.356 palabras únicas, filtrar reviews muy cortas y normalizare el texto.

In [ ]:
!pip install nltk num2words

In [ ]:
import unicodedata
import string    # contiene string.punctuation para los signos de puntuación

import pandas as pd   # DataFrame
from collections import Counter  # para comparar vocabularios antes/después del preprocesado
from num2words import num2words   # para convertir números a palabras

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk import RegexpTokenizer    # tokenizador por expresión regular
from nltk.corpus import stopwords     # stopwords en inglés
from nltk.stem import WordNetLemmatizer  # lematizador basado en WordNet

import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
# Cargo el dataset.
df = pd.read_csv('amazon_sentiment_balanced.csv')
df.head()

,text,rating,label
0,The seller I bought it from sold it to me with...,5.0,1
1,"Ok, here are the reasons this game is BAD!.1. ...",1.0,0
2,"For years now,gamers have said that this is on...",2.0,0
3,When 3D fighters first entered with Virtua Fig...,5.0,1
4,"When I first played Super Mario 64 as a kid, I...",5.0,1


# 2. Pipeline de preprocesado

Construyo la función principal `nltk_cleaner` siguiendo el mismo esquema visto en clase.
Recibe el texto y los objetos necesarios como parámetros: `tokenizer`, `sw_list` y `lemmatizer`.

Pasos que realizo :
1. EliminO acentos con unicodedata, lo sigo utilizando para habituarme cuando tenga texto en castellano.
2. Tokenizo con `RegexpTokenizer` (elimina puntuación)
3. Elimino stopwords en ingles.
4. Convierto a minúsculas y lematizar.
5. Convierto números a palabras con num2words.

In [ ]:
# Instancio los objetos del pipeline.
tokenizer  = RegexpTokenizer(r'\w+')    # Separa por cualquier carácter no alfanumérico.
sw_list    = stopwords.words('english')    # Stopwords en inglés
lemmatizer = WordNetLemmatizer()      # Lematizador de NLTK basado en WordNet.

In [ ]:
def nltk_cleaner(text, tokenizer, sw_list, lemmatizer):
    clean_text = list()


    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')

    # Lowercase ANTES de tokenizar para que las stopwords se reconozcan bien
    # Antes lo puse con el lematizer como en clase y se me colaron "i","the" y otras basurillas.
    text = text.lower()

    #  Tokenizo
    for word in tokenizer.tokenize(text):

        # Elimino stopwords
        if word not in sw_list:

            # Lematizo
            clean_word = lemmatizer.lemmatize(word).strip()

            # Convierto dígitos a palabras
            if clean_word.isdigit():
                clean_word = num2words(clean_word, lang='en')

            clean_text.append(clean_word)

    return ' '.join(clean_text)

In [ ]:
# Pruebo con una review del dataset
review = df.iloc[0]['text']
print('TEXTO ORIGINAL:')
print(review)
print()
print('TEXTO PREPROCESADO:')
print(nltk_cleaner(review, tokenizer, sw_list, lemmatizer))

TEXTO ORIGINAL:
The seller I bought it from sold it to me with brand-new conditions, but what else can I say, it's a black controller that suits me well.

TEXTO PREPROCESADO:
seller bought sold brand new condition else say black controller suit well


In [ ]:
# Pruebo con una frase con números y caracteres extraños para ver todos los pasos
test = 'This game is AMAZING!! I bought 2 copies for my friends :) sooooo worth it!!'
print('ORIGINAL:      ', test)
print('PREPROCESADO:  ', nltk_cleaner(test, tokenizer, sw_list, lemmatizer))

ORIGINAL:       This game is AMAZING!! I bought 2 copies for my friends :) sooooo worth it!!
PREPROCESADO:   game amazing bought two copy friend sooooo worth


In [ ]:
df['text_clean'] = df['text'].apply(
    lambda x: nltk_cleaner(x, tokenizer, sw_list, lemmatizer)
)

df[['text', 'text_clean', 'label']].head()

,text,text_clean,label
0,The seller I bought it from sold it to me with...,seller bought sold brand new condition else sa...,1
1,"Ok, here are the reasons this game is BAD!.1. ...",ok reason game bad one graphic good plus level...,0
2,"For years now,gamers have said that this is on...",year gamers said one greatest game ever made u...,0
3,When 3D fighters first entered with Virtua Fig...,3d fighter first entered virtua fighter impres...,1
4,"When I first played Super Mario 64 as a kid, I...",first played super mario sixty-four kid amazed...,1


In [ ]:
# Elimino reviews que hayan quedado vacías tras el preprocesado y parece que no las había.
df = df[df['text_clean'].str.strip() != ''].reset_index(drop=True)
print(f'Reviews tras eliminar vacías: {len(df)}')

Reviews tras eliminar vacías: 5000


In [ ]:
# Vocabulario ANTES del preprocesado
words_before = []
for text in df['text']:
    words_before.extend(text.lower().split())

vocab_before = Counter(words_before)

# Vocabulario DESPUÉS del preprocesado
words_after = []
for text in df['text_clean']:
    words_after.extend(text.split())

vocab_after = Counter(words_after)

print(f'Vocabulario ANTES:  {len(vocab_before):,} palabras únicas')
print(f'Vocabulario DESPUÉS: {len(vocab_after):,} palabras únicas')
print(f'Reducción: {(1 - len(vocab_after)/len(vocab_before))*100:.1f}%')

Vocabulario ANTES:  64,006 palabras únicas
Vocabulario DESPUÉS: 23,314 palabras únicas
Reducción: 63.6%


In [ ]:
print('Top 20 palabras más frecuentes ANTES:')
print(vocab_before.most_common(20))

Top 20 palabras más frecuentes ANTES:
[('the', 51927), ('and', 25227), ('to', 23637), ('a', 22308), ('of', 18794), ('is', 16849), ('you', 15962), ('i', 15446), ('game', 13393), ('this', 13181), ('it', 12580), ('in', 10834), ('that', 9731), ('for', 8221), ('are', 7431), ('but', 6849), ('have', 6126), ('with', 6064), ('on', 5617), ('was', 5598)]


In [ ]:
print('Top 20 palabras más frecuentes DESPUÉS:')
print(vocab_after.most_common(20))

Top 20 palabras más frecuentes DESPUÉS:
[('game', 23112), ('one', 6249), ('like', 4396), ('get', 4013), ('time', 3689), ('two', 3194), ('play', 3178), ('graphic', 2742), ('good', 2739), ('even', 2482), ('character', 2470), ('first', 2272), ('really', 2254), ('great', 2171), ('fun', 2163), ('level', 2015), ('make', 2015), ('much', 1843), ('thing', 1823), ('would', 1791)]


In [ ]:
# Guardo el dataset preprocesado en un csv.
df.to_csv('amazon_sentiment_preprocessed.csv', index=False)